In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Série simulée avec tendance
np.random.seed(42)
n = 200
time = np.arange(n)
trend = 0.1 * time
noise = np.random.normal(scale=1, size=n)
series = trend + noise

plt.plot(series)
plt.title("Série avec tendance")
plt.show()

# Ajustement ARIMA(p=1, d=1, q=1)
model = ARIMA(series, order=(1,1,1))
fit = model.fit()

print(fit.summary())

# Prévisions
forecast = fit.forecast(10)
plt.plot(series, label="Série")
plt.plot(np.arange(n, n+10), forecast, label="Prévisions", color="red")
plt.legend()
plt.show()


In [0]:
%pip install LDCCropMonitor sqlmodel
%reload_ext autoreload

In [0]:
from LDCDataAccessLayerPy import databricks_init
databricks_init('REMOTESENSING')
from LDCDataAccessLayerPy import database, model, query


In [0]:
from LDCCropMonitor import model

In [0]:
from LDCCropMonitor import query

In [0]:
print(dir(LDCCropMonitor))

In [0]:
LDCCropMonitor.init_prod(read_only=True)

In [0]:
database.init_prod(read_only=True)

In [0]:
from sqlmodel import Session, select
from datetime import datetime, date

In [0]:
indic = model.TemperatureEcmwfDaily
start = datetime(2022, 1, 1)
end = datetime(2022, 12, 31)
with Session(database.engine) as session:
  geo_query = select(model.Geolocation).where(model.Geolocation.gid_ldc == 'LDC_gno_ARG_Corn')
  geo = session.exec(geo_query).first()
  data = query.get_raw_and_aggregated_realised_data(indic, geo, session, start, end)
display(data)

In [0]:
indic = model.SoilMoistureIEcmwfIfsStfcDaily
publication = date.today()
with Session(database.engine) as session:
  geo_query = select(model.Geolocation).where(model.Geolocation.gid_ldc == 'LDC_gno_ARG_Corn')
  geo = session.exec(geo_query).first()
  data = query.get_raw_forecast_data(indic, geo, session, publication)
display(data)

In [0]:
import pandas as pd
with Session(database.engine) as session:
  geom_query = select(model.Geometry.id, model.Geometry.parent_gid_code, model.Geometry.level, model.Geometry.gid_code, model.Geometry.name)
  geom = session.exec(geom_query).all()
df_geom = pd.DataFrame(geom)
df_geom

In [0]:
with Session(database.engine) as session:
  geom_query = select(model.Area.id, model.Area.year, model.Area.geolocation_id, model.Area.geometry_id, model.Area.value).where(model.Area.geolocation_id == geo.id)
  print(geo_query.compile(compile_kwargs={"literal_binds": True}))
  geom = session.exec(geom_query).all()
df_geom = pd.DataFrame(geom)
df_geom